# 六足ロボットで円柱を押す：APIの操作から比較実験まで

六足ロボットが円柱を目標位置まで押す課題で、**観測・行動・報酬の設定、PPOによる学習・評価、結果の比較と考察**を行います。
環境は`reset()`で初期化し、`step()`で動かします。観測・行動・報酬はTensorDictで受け渡します。
Actorは観測から行動を選ぶ方策、Criticは状態の価値を推定するネットワークです。

ロボットの制御は2段階です。今回学習するPPOは、停止または移動方向を選びます。
その指令から18関節を動かす歩行モデルは、学習済みのものを固定して使います。
方向を選ぶPPOのActor・Criticは、ランダムな重みから学習します。

| Step | 操作・実験 | 確認する内容 |
|---|---|---|
| Step 0〜1 | 準備し、APIで環境を動かす | 観測値・行動・1ステップの報酬 |
| Step 2〜7 | 比較実験の結果を読む | 仮説・固定条件・学習曲線・数値・動画・考察 |
| Step 8〜9 | 標準条件を学習し、報酬1項目を変える | 学習したモデルと評価結果 |
| Step 10〜13 | 学習を繰り返して比較し、発展課題と報告へ進む | ばらつき・独自設定・実験記録 |

標準報酬と進捗報酬の重みを変えた条件を、それぞれ3回ずつ学習した比較実験を扱います。
まずStep 7までを順に実行してください。ここまではPPOの学習を実行せずに取り組めます。
Step 9は既定で動作獲得のための本学習を1回実行します。追加実験は各セルのスイッチを有効にしたときに実行します。

## Step 0 — 実行環境を準備する

Colabのランタイムは、コードを実行する計算機のセッションです。
「ランタイム → ランタイムのタイプを変更」で **T4 GPU / ランタイムバージョン2026.07** を選びます。
2026.07のPython 3.12.13・PyTorch 2.11.0は、比較実験の本学習で使用した版です。
選択方法と提供期間は[Colab公式の一覧](https://research.google.com/colaboratory/runtime-version-faq.html)で確認できます。

次のセルは、GitHubのリポジトリ（コードと実験データ）を`git clone`で`PROJECT_DIR`へコピーし、必要なライブラリを準備します。
`prepare_colab`は、必要なライブラリのインストールと描画の準備をまとめた関数です。
同じセッションで再実行した場合は、取得済みのファイルを使います。
更新版を使うときは、GitHubのリンクから開き直し、新しいランタイムで準備してください。

セルは上から順に実行します。自分の予想や考察は、Colabの「＋テキスト」で追加したセルに書きます。
後のセルで使う変数が消えた場合は、準備セルから必要な範囲を再実行してください。

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = "https://github.com/hashimoto-robotics-lab/hexapod-transport-rl.git"
REVISION = "main"
PROJECT_DIR = Path("/content/hexapod_transport_rl")

if not PROJECT_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REVISION,
        REPOSITORY, str(PROJECT_DIR),
    ], check=True)

sys.path.insert(0, str(PROJECT_DIR))
from colab_setup import prepare_colab

prepare_colab(PROJECT_DIR)

## Step 1 — APIでロボットの観測と行動を確かめる

APIは、環境や学習処理を呼び出すためのクラス・関数です。
まず、**環境を作る → 観測を読む → 行動を渡す → 次の状態を読む → 観測を変える**を実行します。

この課題には、六足ロボット1台・半径0.25 m、高さ0.25 m、質量2 kgの円柱・目標があります。
ロボットの中心は円柱の中心から0.65〜1.00 mの周囲に置き、水平な向きも全周からランダムに決めます。
目標は円柱の中心から0.30〜0.60 m、全方向からランダムに決めます。
目標に円柱の中心を近づけ、**位置誤差5 cm未満・円柱速度0.06 m/s未満を0.4秒保持**すると成功です。

`reset()`から成功・転倒などの失敗・40秒の時間切れまでを、1エピソードと呼びます。
円柱の向きは成功判定に含めません。成功条件は、後で報酬を変えても共通です。

TorchRLでの主な処理と、このAPIは次のように対応します。

| TorchRLの処理 | API |
|---|---|
| 環境と入力・行動を決める | `CylinderPushTask`と`make_env()` |
| 観測を受け取り、1ステップ動かす | `env.reset()`と`env.step(td)` |
| 経験収集、GAE（報酬と価値から方策更新用の評価量を計算）、PPOの損失計算と更新 | Step 9の`train()`がまとめて実行 |
| 学習した方策を試す | `run.evaluate()`と`run.video()` |

下ではPyTorch、表を扱うpandas、図を描くmatplotlibと、最初に使うAPIを読み込みます。

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

from hexapod_transport_rl import CylinderPushTask, DiscreteDirections, ObservationTerm

### 1.1 課題を定義し、`reset()`で最初の観測を受け取る

`CylinderPushTask`は、観測・行動・報酬をまとめる設定です。
ここでは位置・速度・向きなど14個の数値を入力にし、停止と8方向の移動から1つを選ぶ課題を作ります。

| 指定 | このセルでの意味 |
|---|---|
| `observations` | 方策へ渡す観測項の並び。`ObservationTerm("baseline")`は標準の14成分のセット名 |
| `actions` | 選べる行動。`count=8`は移動8方向、`include_stop=True`は停止も加える指定 |
| `frame="world"` | 行動を床に固定された世界XY座標で解釈する。機体が回っても軸は変わらない |
| `speed=0.15` | 移動時に要求する速度の大きさ [m/s] |
| `rotation="goal_octant"` | 目標方向に合わせ、方策の入出力を45度刻みで座標変換する設定。Step 3で学習条件と一緒に確認する |

報酬を省略すると標準の報酬を使います。その計算はStep 2で確認します。
`make_env`は、設定から物理シミュレーション環境を作ります。
`backend="cpu"`はCPUで接触や移動を計算する指定、`render_mode="rgb_array"`は表示用の画像を取得する指定です。
画像は動作確認用で、方策には`td["observation"]`の数値を渡します。

`seed`は乱数の初期値です。`env.set_seed(42)`の後に`reset()`すると、初期配置を同じ値で取り出せます。
`reset`の戻り値はTensorDictです。観測の形`(1, 14)`は「独立した世界が1個、各世界の入力が14成分」を表します。
`num_envs`はこの世界の数です。256にすると観測は`(256, 14)`になり、行動も各世界に1つずつ渡します。

In [ ]:
task = CylinderPushTask(
    observations=(ObservationTerm("baseline"),),
    actions=DiscreteDirections(count=8, include_stop=True, frame="world", speed=0.15),
    rotation="goal_octant",
)
env = task.make_env(num_envs=1, backend="cpu", render_mode="rgb_array")
env.set_seed(42)
td = env.reset()
print("方策の入力の形:", tuple(td["observation"].shape))
print("選べる行動の数:", task.actions.action_count)

plt.figure(figsize=(6, 4))
plt.imshow(env.render())
plt.axis("off")
plt.show()

### 1.2 観測の各成分を実際の数値で読む

次の表の「値」は、いま`reset()`した世界の観測です。テンソルの`[0]`で、最初の世界を取り出しています。
`cargo`は円柱、`robot`はロボット、`goal`は目標を意味します。
`cargo_to_robot`は円柱からロボット、`cargo_to_goal`は円柱から目標への世界XYベクトルです。

位置は円柱の直径0.5 m、XY速度は0.2 m/sで割っています。このように一定の基準で数値の大きさを揃える処理を正規化と呼びます。
例えば相対位置が0.25 mなら、位置の入力は`0.25 / 0.5 = 0.5`です。
`robot_yaw`は世界X軸から測った機体の水平角 [rad]、`robot_yaw_rate`はその変化速度 [rad/s]です。
角度の単位radではπ radが180度です。
向きは`[sin(yaw), cos(yaw)]`の2成分で表し、角度が一周する境目でも連続する値にします。
標準の14成分のうち3成分は固定値`0, 1, 0`です。今回の位置課題では、この部分を一定値として扱います。

`env.state`は、位置などを名前で読める物理状態です。方策へ渡す項は`observations`で選びます。
`task.config.width`は円柱の直径 [m]です。位置から計算した相対ベクトルを直径で割り、入力の最初の2成分と照合します。

In [ ]:
observation_names = [
    "cargo_to_robot.x / 0.5", "cargo_to_robot.y / 0.5",
    "cargo_to_goal.x / 0.5", "cargo_to_goal.y / 0.5",
    "固定値 0", "固定値 1",
    "robot_velocity.x / 0.2", "robot_velocity.y / 0.2",
    "cargo_velocity.x / 0.2", "cargo_velocity.y / 0.2",
    "固定値 0", "sin(robot_yaw)", "cos(robot_yaw)", "robot_yaw_rate",
]
display(pd.DataFrame({"入力成分": observation_names, "値": td["observation"][0].tolist()}))

state = env.state
print("位置から計算:", ((state.robot_position - state.cargo_position) / task.config.width)[0])
print("観測の0〜1番:", td["observation"][0, :2])
print("円柱→目標を直径で割った値:", state.cargo_to_goal[0] / task.config.width)
print("観測の2〜3番:", td["observation"][0, 2:4])

### 1.3 行動番号と移動方向を対応させる

この方策は、連続したXY速度を直接出す代わりに、9個の整数から1つを選びます。これを離散行動と呼びます。
`task.actions.directions`で、行動番号に対応する世界XYの方向ベクトルを取得できます。
0は停止、1は+X、2は+X/+Yの斜め、3は+Yで、以後は反時計回りです。
要求速度はこのベクトルに`speed=0.15` m/sを掛けた値です。斜め方向も速度の大きさは同じです。

固定した歩行モデルが移動方向への旋回と18関節の制御を担当し、接触によって円柱が動きます。
**実行後に確認：** 表の0、1、3番は、それぞれどの方向になっていますか？

In [ ]:
action_table = pd.DataFrame(task.actions.directions, columns=["世界X方向", "世界Y方向"])
action_table.index.name = "行動番号"
display(action_table.round(3))

### 1.4 行動を1つ渡して`step()`する

まず`action_id=1`、つまり世界の+X方向を手動で選びます。PPOのActorも同じ形式の整数を出力します。
`td["action"]`に世界ごとに1つの整数を入れ、`env.step(td)`でシミュレーションを0.2秒進めます。
この1回を、ここでの1ステップと数えます。内部では、接触や関節運動をさらに細かい時間刻みで計算しています。

戻り値の`transition["next"]`には、次の観測・報酬・終了判定が入ります。
`done`は成功、転倒などの失敗、時間切れのいずれかでエピソードが終了したかを表します。
`reward_terms`は重みなどを掛けた報酬項ごとの寄与で、合計が`reward`です。
`before`と`after`には、ステップ前後の名前付き状態を保存します。後で報酬の計算確認にも使います。
表示用の`.item()`は、要素が1つのテンソルをPythonの数値に変換します。

要求速度と実測速度を比べてください。初期の機体の向きによって、旋回から始まることもあります。

In [ ]:
action_id = 1  # 0なら停止、3なら世界の+Y方向
before = env.state
td["action"] = torch.tensor([action_id], dtype=torch.int64, device=env.device)
transition = env.step(td)
next_td = transition["next"]
after = env.state

print("次の観測:", next_td["observation"][0])
print("報酬:", next_td["reward"].item(), "終了:", next_td["done"].item())
display(pd.DataFrame({
    "実行前の位置 [m]": before.robot_position[0].tolist(),
    "実行後の位置 [m]": after.robot_position[0].tolist(),
    "要求速度 [m/s]": after.command_velocity[0].tolist(),
    "実測速度 [m/s]": after.robot_velocity[0].tolist(),
}, index=["世界X", "世界Y"]))
display(pd.DataFrame({"報酬への寄与": {
    name: value.item() for name, value in next_td["reward_terms"].items()
}}))

**小演習：** 下の`env.close()`を実行する前に、`env.set_seed(42)`と`td = env.reset()`を実行し、
上の`action_id`を0や3に変えて1.4のコードをもう一度実行してください。同じ配置で要求速度がどう変わりますか？
環境を閉じた後に試す場合は、1.1の環境作成から実行します。

同じエピソードを続けるには、`td = next_td`として次の`action`を入れ、再び`step`します。
`done=True`になったら`reset`で次のエピソードを始めます。
ここまでの手動操作は環境の確認です。Step 9では`train()`が方策による行動選択とPPOの更新を繰り返します。
確認を終えたら、次のセルで環境が使う資源を解放します。

In [ ]:
env.close()

### 1.5 観測をAPIで6成分に変更する

`dataclasses.replace`は、設定の一部を置き換えた新しい設定を作る関数です。元の`task`はそのまま残ります。
`observations`に、円柱→ロボットの位置2成分、円柱→目標の位置2成分、機体の向き2成分を並べます。
APIは指定した順に連結します。`robot_heading`は`[sin(yaw), cos(yaw)]`です。
`scale`は割る基準値で、位置には標準と同じ円柱直径`task.config.width`を指定します。

1.1の回転変換は標準14入力用なので、入力を変えるこの練習では`rotation="none"`で変換を無効にします。
`observation_layout`は、各項が入力の何番目に入ったかを示す一覧です。
**実行前に予想：** 入力の形はどうなりますか？ ロボットや円柱の速度は入力に残りますか？

In [ ]:
from dataclasses import replace

small_task = replace(task, observations=(
    ObservationTerm("cargo_to_robot", scale=task.config.width),
    ObservationTerm("cargo_to_goal", scale=task.config.width),
    ObservationTerm("robot_heading"),
), rotation="none")
small_env = small_task.make_env(num_envs=1, backend="cpu")
small_env.set_seed(42)
small_td = small_env.reset()

display(pd.DataFrame(small_env.observation_layout)[["name", "start", "stop", "scale"]])
print("変更後の入力の形:", tuple(small_td["observation"].shape))
print("変更後の入力:", small_td["observation"][0])
small_env.close()

`start`から`stop`の直前までが、その項の入力です。例えば`cargo_to_goal`は`[:, 2:4]`に入り、全世界の2〜3番目を取り出せます。
入力は`(1, 6)`になり、ロボットと円柱の速度・旋回速度がなくなりました。

**小演習：** `observations`の最後に`ObservationTerm("cargo_velocity", scale=0.2)`を追加し、
1.5のコードを再実行して入力が`(1, 8)`になることを確認してください。
速度の情報を使えると、目標付近でどんな判断がしやすくなるか予想も書いてください。

ここではAPIの操作を練習しました。以降の報酬比較では元の14入力を使います。
入力変更が学習性能に与える影響は、Step 12で別の実験として扱います。

## Step 2 — 報酬を1項目変え、仮説を立てる

Step 1では環境を手で動かしました。次は、報酬の変更がPPOの学習にどう影響するかを考えます。
ここでは、**標準の報酬**と**目標へ近づく報酬の重みを1つ変えた報酬**で学習・評価した比較実験を扱います。
Step 2〜7で結果を確認し、Step 8から同じ手順で学習を実行します。

`CylinderPushTask.baseline()`は、1.1で書いた14入力・9行動・報酬・回転変換の標準設定をまとめて作ります。
`base`が比較の基準、`changed`が変更した設定です。
`rewards`は「報酬項の名前 → 重み」を指定する辞書で、作成後の各項は`RewardTerm`として重みと計算方法を持ちます。
`{**base.rewards, "goal_progress": 40.0}`は、全項をコピーして`goal_progress`だけを上書きする書き方です。

`goal_progress`は、1ステップで円柱が目標に近づいた距離です。
例えば距離が0.30 mから0.28 mになると、重み20の寄与は`20 × 0.02 = 0.4`、重み40では`0.8`です。
離れると負の値になります。他の項には、回り込み、残り距離、経過時間、成功・失敗を使います。

| 項の名前 | 1ステップの計算 | 標準の重み |
|---|---|---:|
| `goal_progress` | 前の目標距離 − 今の目標距離 | 20 |
| `approach_progress` | 前の回り込みコスト − 今の回り込みコスト | 2 |
| `goal_distance` | 今の目標距離 × `dt` | -4 |
| `approach_distance` | 今の回り込みコスト × `dt` | -1 |
| `time` | `dt` | -0.02 |
| `success` | 成功したら1、それ以外0 | 10 |
| `failure` | 転倒などの失敗なら1、それ以外0 | -10 |

`dt`は1ステップのシミュレーション時間0.2秒です。40秒の時間切れは`failure`の減点には含みません。
回り込みコストは、円柱の外周までの距離と、目標の反対側へ回り込む角度をまとめた量です。
目標の反対側とは「ロボット → 円柱 → 目標」の順に並ぶ側です。このコストはその側へ近づくと小さくなります。

**仮説：** `goal_progress`の重みを20から40にすると、同じ学習量で成功率が上がり、終了時の目標距離が小さくなる。
逆に、目標付近で押し続ける可能性も考えてみてください。結果を見る前に、自分の予想と理由を1文で書きます。
次のコードで、重みの違いと、Step 1の同じ状態変化に対する報酬の寄与を確認します。

In [ ]:
base = CylinderPushTask.baseline()
changed = replace(base, rewards={**base.rewards, "goal_progress": 40.0})

weight_table = pd.DataFrame({
    "標準の重み": {name: term.weight for name, term in base.rewards.items()},
    "変更後の重み": {name: term.weight for name, term in changed.rewards.items()},
})
display(weight_table)

contribution_table = pd.DataFrame({
    "標準の寄与": {name: term.compute(before, after).item() for name, term in base.rewards.items()},
    "変更後の寄与": {name: term.compute(before, after).item() for name, term in changed.rewards.items()},
})
display(contribution_table)

## Step 3 — 比較するために学習と評価の条件を揃える

### 3.1 学習seedを揃えた組で比べる

Step 1の`seed`を、PPOの初期重みや行動の乱数にも使います。これを学習seedと呼びます。
実験例では学習seedを42・43・44とし、各条件を毎回ゼロから学習しました。
同じseedの標準・変更条件は同じランダムな初期重みで始めます。異なるseedでは初期重みも異なります。
学習が進んで選ぶ行動が変われば、その後に集まる経験も変わります。

PPOの更新にはAdamという最適化手法を使います。Adamが持つ過去の更新情報も、各学習の開始時に新規にします。
この組を3つ作ることで、報酬変更の効果が、1回だけの偶然に左右されていないかを調べます。

### 3.2 易しい配置から範囲を広げる

学習中の開始配置を易しいものから難しいものへ変える方法を、カリキュラム学習と呼びます。
`PushCurriculum()`は、開始配置を次の予定で変えます。
「円柱の後ろ」は、目標の反対側です。最初はそこから押し、最後はStep 1の全方向ランダムの課題にします。

| 総学習量の範囲 | 練習する開始配置 | 学習率 |
|---|---|---:|
| 0〜20% | 円柱の後ろ。目標は世界X軸から45度、距離0.15〜0.25 m | 0.0003 |
| 20〜35% | 後ろから開始。目標は全方向・距離0.30〜0.60 m | 0.0003 |
| 35〜65% | 円柱の周囲から開始し、機体は円柱にほぼ向く | 0.0003 |
| 65〜80% | 開始時の機体の向きを、円柱方向から±90度まで広げる | 0.0003 |
| 80〜100% | 位置・向き・目標が全方向ランダム | 0.00003 |

学習率は、PPOが重みを更新するときの変化量を調整する値です。
段階が変わっても同じActor・CriticとAdamの記録を保持し、1回の`train()`の中で学習を続けます。
開始配置の変更は次のエピソードから適用します。報酬・観測・行動・物理・成功条件は全段階で共通です。
両条件でこの予定と総学習量を固定し、報酬の重み1つの影響を比べます。

### 3.3 学習・検証・最終評価を分ける

学習中の経験は、重みを更新するために使います。
途中の方策を同じ配置で試して選ぶ作業を**検証**、選択が終わったモデルの成績を別の配置で測る作業を**最終評価**と呼びます。
検証・評価では重みを更新せず、各観測でActorが最も高い確率を付けた行動を使います。

| 用途 | 使うseedと回数 | 何に使うか |
|---|---|---|
| 学習 | 学習seed 42・43・44 | 経験を集め、PPOの重みを更新 |
| 検証 | 検証seed 131000、256エピソード | 学習途中のモデルから報告用を選ぶ |
| 実験例の最終評価 | 評価seed 840000、1,024エピソード | 選んだモデルの成績を測る |
| Step 9以降の最終評価 | 評価seed 940000、1,024エピソード | 新しく学習したモデルを測る |

評価seedは初期配置を作る乱数に使います。同じ用途では、全条件・全学習seedのモデルを共通の配置で試します。
検証と最終評価は、最初から全方向ランダムの課題です。練習中の易しい配置での成功率とは測っている対象が異なります。

保存した重みや設定のファイルを**チェックポイント**と呼びます。
規定の総学習量を終え、最後の全方向ランダムの段階の検証で、目標方向別などの最低成功率を優先して`best.pt`を選びます。
これにより方向による不得意も確認します。最後の更新は`latest.pt`に残ります。
最終評価は選択した後に行い、その結果を使ってモデルを選び直しません。

### 3.4 学習量と座標変換を確認する

1サンプルは、1つの世界で1回の行動と次の状態を得る経験です。
`num_envs`は並列世界数、`horizon`は1回の収集で各世界を進めるステップ数です。
この収集のまとまりをロールアウトと呼び、1回で`num_envs × horizon`サンプルが集まります。
実験例の`512 × 64 = 32,768`サンプルを300回集めると、`steps = 9,830,400`です。
各収集後には、集めた経験でPPOを複数回更新します。1サンプル、1ロールアウト、1回の勾配更新は異なる単位です。

`rotation="goal_octant"`では、目標方向が45度付近に来るよう、45度の整数倍だけ位置・速度・向きを回して、ActorとCriticへ入力します。
例えば世界の+Y方向（90度）の目標なら、45度回した座標で扱います。1.2で読んだ数値は、この変換前の世界座標です。
Actorが選んだ方向は世界座標へ戻します。似た配置を回転させて共通に扱う、この仕組みを回転対称化と呼びます。
円柱と8方向の行動に合わせた前処理で、標準の14入力・停止＋世界8方向の両条件で共通に使います。

実験の設定は`protocol.json`に保存されています。JSONは辞書や配列を保存する形式です。
下の`protocol`は、このファイルをPythonの辞書として読み込んだものです。

In [ ]:
STUDY_DIR = PROJECT_DIR / "results"
protocol = json.loads((STUDY_DIR / "protocol.json").read_text())

display(pd.DataFrame({
    "固定した項目": ["学習seed", "各seedの総サンプル数", "並列世界数 num_envs", "収集長 horizon", "最終評価エピソード数", "変更した項目"],
    "値": [str(protocol["training_seeds"]), f"{protocol['training']['steps']:,}",
           str(protocol["training"]["num_envs"]), str(protocol["training"]["horizon"]),
           str(protocol["evaluation"]["episodes"]), "goal_progress: 20 → 40"],
}))

## Step 4 — 学習曲線で、学び方の変化を読む

ここから、標準報酬と変更後の報酬で学習した実測結果を確認します。新たな学習は実行しません。
`Image`で保存済みの図を表示し、`pd.read_csv`で元データを表として読み込みます。CSVは行と列で数値を保存する形式です。
図の`baseline`は標準の重み20、`progress40`は変更後の重み40を表します。
ここでの`baseline`は条件の名前です。Step 1の観測セット名と同じ綴りを使っています。

左は全方向ランダムの検証成功率、右はエピソード終了時の円柱中心と目標中心の距離です。
成功率は高いほど、距離は小さいほど良いと読みます。横軸はStep 3で数えた総学習サンプル数です。
横軸0は、PPOの学習前のランダムな方策です。

細線は3つの学習seedごとの成績、太線はその平均です。帯は3本のばらつきの大きさを表す標準偏差です。
縦の点線は開始配置の切替、灰色の範囲は最後の全方向ランダムの学習段階です。
点は、検証で報告用に選んだチェックポイントの時点です。

**読む順序：** 最初の成功率、学習途中の変化、最後の段階の成績、3本の違いを順に見てください。
「増え続けているか」「同じサンプル数でどちらが高いか」を数値や横軸の位置で書き留めます。
検証の課題は全時点で同じです。開始配置を切り替えた後の低下は、検証配置を難しくしたことによる低下ではありません。

元データの`validation_success_rate`と`validation_distance_m`は検証時の成功率と平均距離、`selected`は選ばれた時点かどうかを表します。

In [ ]:
from IPython.display import Image

display(Image(filename=str(STUDY_DIR / "learning_curves.png"), width=950))
learning_curve = pd.read_csv(STUDY_DIR / "learning_curve.csv")
display(learning_curve[["condition", "seed", "steps", "validation_success_rate", "validation_distance_m", "selected"]].tail(6))

## Step 5 — 最終評価を、学習ごとのばらつきと一緒に読む

ここではStep 3で選んだモデルを、最終評価用の1,024配置で試した結果を見ます。
最初の表は「1条件・1学習seed」が1行で、全6行あります。列は次の意味です。

| 列名 | 意味・単位 |
|---|---|
| `condition` / `seed` | 報酬条件の名前 / 学習seed |
| `successes` / `episodes` | 成功した回数 / 評価エピソード数 |
| `success_rate` | 成功回数 ÷ 評価回数。0〜1で、0.98なら98% |
| `mean_distance_m` | 成功・失敗を含む終了時の目標距離の平均 [m] |
| `selected_steps` | 検証で選んだモデルを保存した時点の学習サンプル数 |
| `mean_episode_seconds` | 成功・失敗を含むエピソード終了までの平均シミュレーション時間 [秒] |

次の集計表は、各条件の3回の学習をまとめます。`mean_`は平均、`std_`は標本標準偏差です。
標本標準偏差は、3つの学習結果が平均からどの程度離れているかを、分母を`3 - 1`にして計算した値です。
距離のばらつきは、各学習seedで求めた平均距離3個から計算します。
3,072配置で1つのモデルを試した結果や、3,072回学習した結果としては扱いません。

`mean_training_seconds`は学習にかかった実時間の平均で、エピソード内の時間とは別です。
下のコードでは成功率と距離の平均・標準偏差を、両方表示します。

In [ ]:
results = pd.read_csv(STUDY_DIR / "runs.csv")
summary = json.loads((STUDY_DIR / "summary.json").read_text())
display(results[["condition", "seed", "successes", "episodes", "success_rate", "mean_distance_m", "selected_steps", "mean_episode_seconds"]])
display(pd.DataFrame(summary["conditions"]).T[[
    "mean_success_rate", "std_success_rate", "mean_distance_m", "std_distance_m", "mean_training_seconds",
]])
display(Image(filename=str(STUDY_DIR / "paired_results.png"), width=850))

図の線は、同じ学習seedの標準・変更条件を結びます。平均に加え、全3組で同じ方向に変わったかを確認します。
`trained_steps`は、選んだモデルの時点を含む総学習量です。`selected_steps`の時点が早くても、実際には規定の総量を学習しています。

**小演習：** 各seedの「変更−標準」の成功率の差を求めてください。
成功率0.01の差は1パーセントポイントです。例えば0.98から0.91への変化は−7ポイントです。
全3組の方向が揃っているか、seedごとの変化量は同じ程度かを説明します。

実時間には全段階の学習と途中の検証を含みます。測定した計算機はRTX A6000の共用ワークステーションです。
GPUには別処理の負荷もあるため、報酬による計算速度の差としては解釈しません。Colab T4での本学習時間は未測定です。

## Step 6 — 共通の初期配置の動画で、動作を読む

数値だけでは、近づき方や押し方、止まり方は分かりません。次は動作を見ます。
動画は学習seed42のモデルを、両条件で同じ初期配置から動かしたものです。
この配置を決める動画seedは850000で、結果を見る前に決めた値です。
`Video`はMP4ファイルをノートブック内で再生する関数です。

**見るポイント：** 円柱へ近づくか、目標の反対側へ回り込むか、目標付近で減速・停止するか。
見えた動作を、どの時刻に起きたかと一緒に書きます。1本の動画は1配置の例なので、成績の判断にはStep 5の全評価も使います。

物理シミュレーションにはMuJoCoを使っています。数値の集計はGPUで多数の世界を計算するMuJoCo Warp、
動画はCPU版のMuJoCoで描画しています。この計算方法の指定が`backend`です。
計算法の差で動作がずれる場合があるため、CPU動画の成功をGPU評価表の成功数へ追加しません。

In [ ]:
from IPython.display import Video

for condition in ("baseline", "progress40"):
    print(condition)
    display(Video(str(STUDY_DIR / "videos" / f"{condition}.mp4"), embed=True, width=640))

## Step 7 — 事実と解釈を分け、考察の例と照らし合わせる

Step 2の自分の予想を読み直し、先に次の3点を書いてください。

1. **事実：** 成功率・距離・各seedの差はどう変わったか。表や図の数値を根拠にする。
2. **解釈：** その結果は仮説を支持するか。原因について、何が考えられるか。
3. **限界：** 今回の記録で確かめていないことは何か。

例えば「終了時の距離が大きかった」は数値で確かめられる事実です。
「目標を通り過ぎたからだ」は原因の解釈で、軌跡など追加の根拠が必要です。
下のセルで比較実験の考察を表示し、自分の説明と照らし合わせます。
`Markdown`は、ファイルに保存した見出しや文章を表示するために使います。

In [ ]:
from IPython.display import Markdown

display(Markdown((STUDY_DIR / "discussion.md").read_text()))

## Step 8 — 実験を設計し、設定を保存する

ここからは、現在接続しているランタイムでPPOを学習します。
まず標準条件を再現し、その後に報酬1項目を変えます。仮説・変更する項目・固定する条件・判定指標を、実行前にテキストセルへ書きます。
実験例を再現する場合は重み40、新しい仮説を試す場合は例えば重み30を選び、`MY_HYPOTHESIS`と`PROGRESS_WEIGHT`を書き換えます。

### 実行量を選ぶ

`PROFILE`はこのノートブック用の設定名です。`"research"`は押す動作を学ぶ規定量の本学習、
`"demo"`はAPIが動くかを確かめるための短縮実行です。既定は`"research"`です。
短縮実行は1エピソード2秒・総80サンプルなので、押す動作の獲得や成功率比較には使いません。
実験例の標準条件はRTX A6000で1学習あたり平均約38分でした。Colab T4での所要時間は未測定です。

`options`は`train()`へ渡す設定の辞書です。`**options`はその中身を関数のキーワード引数として渡す書き方です。
`protocol`から実験例の学習条件を読み、次の意味で使います。

| 引数 | 意味 | 本学習の値 |
|---|---|---:|
| `steps` | 全世界を合計した学習サンプル数 | 9,830,400 |
| `num_envs` | 独立した並列世界の数 | 512 |
| `horizon` | 1回の収集で各世界を進めるステップ数 | 64 |
| `validation_envs` | 1回の検証で試すエピソード数 | 256 |
| `validate_every` | 何ロールアウトおきに検証するか | 10 |
| `settings` | PPOの更新回数・ミニバッチ・学習率などを持つ`PPOSettings` | 実験例と同じ |
| `initial_checkpoint` | 学習の出発点。`None`ならランダムな重みから新規作成 | `None` |
| `checkpoint_selection` | 評価・動画に使う保存モデルの選び方 | `"best"` |
| `curriculum` | 開始配置を変える予定 | Step 3の`PushCurriculum()` |

`PPOSettings`の`epochs`は収集した経験を何巡して更新に使うか、`minibatch_size`は1回の勾配更新で使う経験数です。
比較する条件では、実験例と同じ値に揃えます。

このAPIの既定の`backend="auto"`は、GPUが使えるときはWarp、使えないときはCPUを選びます。
本学習にはGPUを使い、比較する条件では同じ計算法を使います。

`WORK_DIR`は今回の実験の保存先です。下のセルを再実行すると、時刻を付けた新しい保存先で実験を始めます。
`my_base`が標準、`my_task`が報酬変更後の課題です。仮説と条件は`experiment_plan.json`に保存します。
まず`PROFILE`・仮説・重みを決め、次のセルを実行してください。

In [ ]:
from datetime import datetime
from hexapod_transport_rl import PPOSettings, PushCurriculum, train

PROFILE = "research"  # 短縮した動作確認は "demo"
MY_HYPOTHESIS = "進捗の重みを40にすると、同じ学習量で成功率が上がる"
PROGRESS_WEIGHT = 40.0

if PROFILE == "demo":
    my_base = replace(base, config=replace(base.config, episode_seconds=2.0))
    options = dict(steps=80, num_envs=2, horizon=2, validation_envs=2,
                   validate_every=5, settings=PPOSettings(epochs=1, minibatch_size=4))
    seeds, evaluation_episodes = (42, 43), 4
else:
    my_base = base
    options = {**protocol["training"], "settings": PPOSettings(**protocol["ppo"])}
    seeds, evaluation_episodes = tuple(protocol["training_seeds"]), protocol["evaluation"]["episodes"]

options.update(initial_checkpoint=None, checkpoint_selection="best", curriculum=PushCurriculum())
my_task = replace(my_base, rewards={**my_base.rewards, "goal_progress": PROGRESS_WEIGHT})
WORK_DIR = PROJECT_DIR / "runs" / datetime.now().strftime("tutorial_%Y%m%d_%H%M%S_%f")
WORK_DIR.mkdir(parents=True)
plan = {
    "hypothesis": MY_HYPOTHESIS, "profile": PROFILE,
    "changed_parameter": "goal_progress", "baseline_weight": 20.0,
    "changed_weight": PROGRESS_WEIGHT, "training_seeds": list(seeds),
    "evaluation_seed": 940000, "initial_checkpoint": None,
    "steps": options["steps"], "initialization": "random actor and critic",
}
(WORK_DIR / "experiment_plan.json").write_text(json.dumps(plan, ensure_ascii=False, indent=2))
print(MY_HYPOTHESIS)
print("保存先:", WORK_DIR)

display(pd.DataFrame({"今回使う値": {
    key: options[key] for key in ("steps", "num_envs", "horizon", "validation_envs", "validate_every")
}}))

## Step 9 — 学習し、別の配置で評価し、動画で確認する

Step 8の設定を使い、**学習 → 最終評価 → 動画**を順に実行します。
`train(my_base, 保存先, seed=42, **options)`が標準条件の学習です。
内部では環境を並列に動かし、観測から行動を選んで経験を集め、PPOでActorとCriticを更新します。

`initial_checkpoint=None`なので方向選択のPPOはランダム初期化され、固定歩行モデルを使って学習します。
`PushCurriculum()`で開始配置を広げ、検証で保存モデルを選びます。
戻り値の`run`は`TrainingRun`という学習結果の窓口で、保存先と選択したモデルへのパスを持ちます。
セルの再実行は新たな学習を1回追加します。まず学習を完了させ、表示される保存先を確認してください。

In [ ]:
run = train(my_base, WORK_DIR / datetime.now().strftime("baseline_%H%M%S_%f"), seed=42, **options)
print("学習の保存先:", run.directory)
print("評価に使うモデル:", run.checkpoint)
elapsed = json.loads((run.directory / "run.json").read_text())["seconds"]
print(f"標準条件の総学習時間: {elapsed / 60:.1f} 分")

### 9.1 学習したモデルを最終評価する

`run.evaluate()`は、選択済みの`run.checkpoint`を読み、重みを更新せずに試します。
`episodes`は評価する初期配置の数、`seed=940000`はその配置を作る評価seedです。
学習seed42や検証seed131000とは用途が異なります。報酬変更条件も同じ940000で試します。
`report`は結果の辞書です。成功回数を`episode_count`で割り、成功率を計算します。
`one_seed_records`には、後で比較・作図するために今回の1回の学習結果を残します。

In [ ]:
report = run.evaluate(episodes=evaluation_episodes, seed=940000)
one_seed_records = [{
    "condition": "baseline", "seed": 42, "checkpoint": str(run.checkpoint),
    "success_rate": report["successes"] / report["episode_count"],
    "mean_distance_m": report["mean_distance_m"],
}]
display(pd.DataFrame(one_seed_records)[["condition", "success_rate", "mean_distance_m"]])

### 9.2 1エピソードの動画を保存する

`run.video()`も選択済みのモデルを使います。`seed=950000`は動画の初期配置を決める値です。
この動画はCPUで計算する1配置の観察例です。最終評価の全体の成績と合わせて読みます。
短縮実行の場合は、保存や再生ができることを確認してください。

In [ ]:
run.video(WORK_DIR / "baseline.mp4", seed=950000)
display(Video(str(WORK_DIR / "baseline.mp4"), embed=True, width=640))

### 9.3 報酬1項目を変え、同じ出発点から学習する

Step 8で決めた`my_task`は、`my_base`の`goal_progress`の重みだけを変えた課題です。
次の`RUN_REWARD_EXPERIMENT`は、この追加実験を行うかを決めるスイッチです。
`False`なら標準条件だけで次へ進み、`True`にすると変更条件を新たに1回学習します。

標準モデルの重みは引き継がず、同じ学習seed42のランダム初期重みから学習します。
評価seed940000、動画seed950000、総学習量も標準条件と揃えます。
Step 8で設定を決め、標準条件を学習した後は、このセルだけを`True`で実行すれば追加できます。
表は2条件×1学習seedになります。まだ学習間のばらつきは分かりません。
自分の実験では、変更後の重みに対応する条件名を`changed`と記録します。

In [ ]:
RUN_REWARD_EXPERIMENT = False  # 標準条件の後、このセルだけTrueにして実行

if RUN_REWARD_EXPERIMENT:
    reward_run = train(my_task, WORK_DIR / datetime.now().strftime("reward_weights_%H%M%S_%f"), seed=42, **options)
    reward_report = reward_run.evaluate(episodes=evaluation_episodes, seed=940000)
    one_seed_records = [one_seed_records[0], {
        "condition": "changed", "seed": 42, "checkpoint": str(reward_run.checkpoint),
        "success_rate": reward_report["successes"] / reward_report["episode_count"],
        "mean_distance_m": reward_report["mean_distance_m"],
    }]
    reward_run.video(WORK_DIR / "reward_weights.mp4", seed=950000)
    display(Video(str(WORK_DIR / "reward_weights.mp4"), embed=True, width=640))

    display(pd.DataFrame(one_seed_records)[["condition", "success_rate", "mean_distance_m"]])

else:
    print("追加実験は未実施です。現在の結果は標準条件のみです。")

## Step 10 — 学習seedを変えて、結果のばらつきを調べる

Step 9の1組だけでは、報酬変更の効果が他の初期重みでも現れるか分かりません。
`compare()`は「条件名 → 課題」の辞書と学習seedの一覧を受け取り、各組をゼロから学習・評価します。
`RUN_MULTISEED_STUDY=True`にすると、本学習では2条件×3 seedの計6回を新たに実行します。
短縮設定では2条件×2 seedの動作確認です。Step 9の結果は再利用しません。

`evaluation_episodes`と`evaluation_seed`は最終評価の回数と配置の乱数です。
`compare()`の戻り値は、各学習の`records`と、条件ごとの集計`summary`を持つ辞書です。
保存先の`comparison.csv`には各学習の数値が入ります。

`False`のままならStep 9までの結果を使って、下の集計と作図を練習します。
学習が1回だけの条件は、標準偏差を計算できないので「未計算」と表示します。
`groupby("condition")`で同じ条件の行をまとめ、`mean`と`std`で平均と標準偏差を計算します。
表の`training_runs`は、各条件で実行した学習seedの数です。

In [ ]:
from hexapod_transport_rl import compare

RUN_MULTISEED_STUDY = False  # 2条件×3 seedの実験を行うときTrue

comparison = None
if RUN_MULTISEED_STUDY:
    comparison_dir = WORK_DIR / datetime.now().strftime("comparison_%H%M%S_%f")
    comparison = compare(
        {"baseline": my_base, "changed": my_task}, comparison_dir,
        seeds=seeds, evaluation_episodes=evaluation_episodes, evaluation_seed=940000,
        **options,
    )
    display(pd.read_csv(comparison_dir / "comparison.csv"))
    display(pd.DataFrame(comparison["summary"]).T)

experiment_records = comparison["records"] if comparison else one_seed_records
my_results = pd.DataFrame(experiment_records)
my_results.to_csv(WORK_DIR / "experiment_results.csv", index=False)
result_summary = my_results.groupby("condition").agg(
    training_runs=("seed", "nunique"),
    mean_success_rate=("success_rate", "mean"),
    std_success_rate=("success_rate", "std"),
    mean_distance_m=("mean_distance_m", "mean"),
    std_distance_m=("mean_distance_m", "std"),
)
summary_display = result_summary.fillna("未計算（学習1回）")
display(summary_display)
summary_display.to_csv(WORK_DIR / "experiment_summary.csv")

### 10.1 学習曲線と、同じseed同士の差を描く

保存された学習記録から、検証成功率の曲線と最終評価の比較図を作ります。
`initial_evaluation.json`は学習前、`validations.json`は各検証時点の結果です。
横軸のサンプル数は「ロールアウト番号 × `num_envs` × `horizon`」で計算しています。

最初の図は各モデルの検証成功率、次の図は最終評価の同じ学習seed同士を結んだものです。
Step 9で標準条件だけを実行した場合は、1条件の点と曲線になります。
複数seedを実行していないときは、繰り返し学習した場合のばらつきについて結論を書かず、追加実験を計画します。
図と元データは`WORK_DIR`に保存します。短縮実行の図は記録・描画の確認用です。

In [ ]:
curve_rows = []
for record in experiment_records:
    directory = Path(record["checkpoint"]).parent
    initial = json.loads((directory / "initial_evaluation.json").read_text())
    curve_rows.append({
        "condition": record["condition"], "seed": record["seed"], "steps": 0,
        "success_rate": initial["successes"] / initial["episode_count"],
    })
    validations = json.loads((directory / "validations.json").read_text())
    for entry in validations:
        value = entry["report"]
        curve_rows.append({
            "condition": record["condition"], "seed": record["seed"],
            "steps": entry["rollout"] * options["num_envs"] * options["horizon"],
            "success_rate": value["successes"] / value["episode_count"],
        })
my_curves = pd.DataFrame(curve_rows)
my_curves.to_csv(WORK_DIR / "experiment_learning_curves.csv", index=False)
display(my_curves.head())

### 10.2 検証成功率の変化を描く

上で作った表を、条件と学習seedごとに分けて線にします。
`Training samples`は学習サンプル数、`Validation success rate`は検証成功率です。
同じ横軸の位置で比較し、学習の進み方とseedによる違いを読みます。

In [ ]:
for (condition, seed), group in my_curves.groupby(["condition", "seed"]):
    plt.plot(group["steps"], group["success_rate"], label=f"{condition}, seed {seed}")
plt.xlabel("Training samples")
plt.ylabel("Validation success rate")
plt.ylim(0, 1)
plt.legend()
plt.savefig(WORK_DIR / "experiment_learning_curves.png", bbox_inches="tight")
plt.show()


### 10.3 最終評価の同じ学習seedを結ぶ

左に成功率、右に終了時の平均距離を描き、同じseedの条件同士を線で結びます。
成功率は上がるほど、距離は下がるほど改善です。各seedの線の向きと変化量を比較してください。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for seed, group in my_results.groupby("seed"):
    labels = list(my_results["condition"].unique())
    pair = group.set_index("condition").reindex(labels)
    for ax, metric in zip(axes, ("success_rate", "mean_distance_m")):
        ax.plot(range(len(labels)), pair[metric], "o-", label=f"Seed {seed}")
        ax.set_xticks(range(len(labels)), labels)
        ax.set_ylabel(metric)
axes[0].set_ylim(0, 1)
axes[1].legend()
fig.tight_layout()
fig.savefig(WORK_DIR / "experiment_paired_results.png")
plt.show()

## Step 11 — 発展：独自の報酬関数を追加する

Step 9までは既存項の重みを変えました。次は「目標10 cm以内で円柱が速く動くほど減点」を新しく定義します。
狙いは、目標付近で速度を抑える判断を学びやすくすることです。効果があるかは学習後に確かめます。

`RewardTerm`は、報酬の関数・重み・時間の扱いをまとめる設定です。
関数`near_goal_speed(before, after)`は前後の`PushState`（Step 1の`env.state`）を受け取ります。
`after.cargo_speed.square()`は現在の円柱速度の二乗、`after.goal_distance < 0.10`は近いときだけ真になる条件です。
掛け算すると、近いときは速度の二乗、遠いときは0になります。

各状態のベクトルは`(B, 2)`、距離や速度の大きさは`(B,)`です。`B`は並列世界の数を表します。
関数の戻り値は世界ごとに1つ、形`(B,)`のテンソルです。
バッチを保つため、関数内では`.item()`や全世界で1つにまとめる`.sum()`を使いません。
`weight=-2.0`で減点にし、`per_second=True`で1ステップの時間`dt`も掛けます。

`%%writefile`はセルの内容をPythonファイルへ書くColabのコマンドです。
ファイル直下に関数を定義すると、学習記録と一緒に関数のソースも保存して後から読み直せます。
`TASK`は`load_task_file()`が読み取る課題設定の変数名です。観測・行動・他の報酬は標準設定をコピーします。

In [ ]:
%%writefile {WORK_DIR}/my_reward.py
from dataclasses import replace
from hexapod_transport_rl import CylinderPushTask, RewardTerm

def near_goal_speed(before, after):
    return after.cargo_speed.square() * (after.goal_distance < 0.10)

baseline = CylinderPushTask.baseline()
TASK = replace(baseline, rewards={
    **baseline.rewards,
    "near_goal_speed": RewardTerm(near_goal_speed, weight=-2.0, per_second=True),
})

In [ ]:
from hexapod_transport_rl import load_task_file

custom_task = load_task_file(WORK_DIR / "my_reward.py")
custom_task = replace(custom_task, config=my_base.config)

# 物理計算を進めず、距離5 cm・速度0.10 m/sの状態を計算確認用に作る
example_state = replace(after,
    goal_distance=after.goal_distance.new_tensor([0.05]),
    cargo_velocity=after.cargo_velocity.new_tensor([[0.10, 0.0]]),
)
custom_contribution = custom_task.rewards["near_goal_speed"].compute(before, example_state)
print("独自項の寄与（-2 × 0.10² × 0.2 = -0.004）:", custom_contribution)

上の計算確認は、関数が使う距離と速度だけを置き換えた例です。`after`や実際の環境は変えていません。
`goal_distance`を`0.15`へ変えると寄与が0になることも確かめてください。
`config=my_base.config`は、短縮・本学習の制限時間などをStep 8と揃えるための指定です。

独自報酬を学習する場合は、次のスイッチを`True`にします。
この条件もランダム初期化から学習し、同じ評価配置で試します。
結果は専用の保存先に残ります。Step 10の報酬重み比較とは別の問いとして報告します。

In [ ]:
RUN_CUSTOM_REWARD = False  # 独自報酬を学習するときTrue

if RUN_CUSTOM_REWARD:
    custom_run = train(custom_task, WORK_DIR / datetime.now().strftime("custom_reward_%H%M%S_%f"), seed=42, **options)
    custom_report = custom_run.evaluate(episodes=evaluation_episodes, seed=940000)
    print("保存先:", custom_run.directory)
    print("成功:", custom_report["successes"], "/", custom_report["episode_count"])
    print("最終距離 [m]:", custom_report["mean_distance_m"])

## Step 12 — 発展：観測か行動を1つ変えて比較する

Step 1.5では、観測を減らしたときの入力の形を確認しました。次は、その変更が学習性能にどう影響するかを比べます。
観測を減らすと利用できる情報が変わり、行動を減らすと選べる移動方向が変わります。
ここでは報酬を標準に固定し、次のいずれか1つを選びます。

| 設定名 | 変更する内容 |
|---|---|
| `"observation"` | 14入力から位置と向きの6入力へ変更 |
| `"action"` | 停止＋8方向の9択から、停止＋4方向の5択へ変更 |
| `"continuous"` | 整数の選択から、[-1, 1]の連続したXYの2出力へ変更 |

連続行動の`ContinuousVelocity()`は、XYの大きさを1以下に調整して`speed`を掛けます。
標準の回転変換は14入力・停止＋世界8方向専用なので、この比較では基準の`scratch_base`も`rotation="none"`にします。
これにより、比較する2条件で座標変換も揃います。
実験例で得た成功率98.80%は回転変換ありの標準条件なので、この新しい基準条件の成績は改めて測ります。

観測を削除すると、ネットワークの入力層の大きさも変わります。
情報の有無だけを調べたい場合は、同じ観測一覧で一部の`ObservationTerm`を`enabled=False`にしてゼロを入れる方法もあります。
具体例は[APIの観測の説明](https://colab.research.google.com/github/hashimoto-robotics-lab/hexapod-transport-rl/blob/main/cylinder_pushing.ipynb#scrollTo=api-reference)にあります。

In [ ]:
from hexapod_transport_rl import ContinuousVelocity

scratch_base = replace(my_base, rotation="none")
observation_task = replace(scratch_base, observations=(
    ObservationTerm("cargo_to_robot", scale=scratch_base.config.width),
    ObservationTerm("cargo_to_goal", scale=scratch_base.config.width),
    ObservationTerm("robot_heading"),
))
action_task = replace(scratch_base, actions=DiscreteDirections(count=4, include_stop=True))
continuous_task = replace(scratch_base, actions=ContinuousVelocity())

`INPUT_CHANGE`で試す変更を1つ選びます。次のセルは`False`なら学習を追加しません。
`True`にすると、Step 10と同じ`compare()`で基準と変更条件をゼロから学習します。
本学習では2条件×3 seedで6回、短縮設定では2条件×2 seedで4回を追加します。
新しい観測・行動に同じ学習量が十分かは、検証曲線と最終評価の両方で確認してください。

In [ ]:
RUN_INPUT_EXPERIMENT = False
INPUT_CHANGE = "observation"  # "action" または "continuous" も選べる

input_task = {"observation": observation_task, "action": action_task, "continuous": continuous_task}[INPUT_CHANGE]
if RUN_INPUT_EXPERIMENT:
    input_dir = WORK_DIR / datetime.now().strftime("input_comparison_%H%M%S_%f")
    input_comparison = compare(
        {"reference": scratch_base, "modified": input_task}, input_dir,
        seeds=seeds, evaluation_episodes=evaluation_episodes, evaluation_seed=940000,
        **options,
    )
    display(pd.read_csv(input_dir / "comparison.csv"))
    print("観測・行動の比較の保存先:", input_dir)

## Step 13 — 実際に行った実験を考察し、記録を保存する

Step 9〜10で行った報酬重みの実験について、次をテキストセルに書きます。
独自報酬や観測・行動の発展課題は、その保存先の結果を使って別の報告を作ります。

1. 結果を見る前の仮説と、変更した1項目。
2. 実行した全学習seedの成功率・距離・学習曲線。複数学習なら平均と標準偏差。
3. 仮説と一致した点・一致しなかった点。根拠になる数値と図・動画。
4. 今回の学習量、学習回数、初期配置の範囲で、まだ言えないこと。
5. 次に確かめる1項目とその理由。

標準条件しか実行していない場合は、報酬の比較は未実施です。標準の動作と成績を確認し、追加実験を計画します。
短縮設定で得た数値はAPIの動作確認として報告します。
近づくが押さない、目標を通り過ぎる、seedで優劣が逆になるなどの結果には、
次の対応表も使ってください。
報酬の重みを変えると総報酬の尺度も変わるため、総報酬が増えただけで性能が上がったとは判断しません。

下の`REPORT`は実験記録のひな形です。`[ ]`の記入欄に結果・考察・限界・次の実験を書き、セルを再実行すると`report.md`に保存されます。
設定は`task.json`、独自関数は`task_sources/`、使用ライブラリの版は`environment.json`に残ります。
これらをチェックポイントと一緒に保管すると、どの条件で学習した結果かを追えます。

### 動作を見て、次に調べる項目を決める

| 見えた結果 | 次に確認するもの |
|---|---|
| 全条件で成功率0 | 短縮設定か本学習か、実際のサンプル数、検証曲線、終了時の距離 |
| 円柱へ近づかない | 相対位置の向きと座標系、接近に関する報酬項、選択した行動 |
| 近づくが押さない | 接触しているか、接近だけで報酬を得ていないか、目標への進捗 |
| 目標を通り過ぎる | 目標付近の速度、停止を選べるか、成功の保持時間 |
| 短時間で失敗して終わる | 転倒などの失敗理由、終了時刻、継続コストと失敗時の減点 |
| 総報酬は増えたが成功率は下がった | 項ごとの報酬の寄与、成功条件との関係 |
| seedで優劣が逆になる | ばらつきの大きさ。追加する学習seed数を決め、全条件を同じ数だけ試す |

表の項目は、原因を確かめるための手掛かりです。今回の記録で分かった事実と、次に検証する説明を分けて書きます。

In [ ]:
comparison_status = "標準と変更条件を実行済み" if "changed" in set(my_results["condition"]) else "比較未実施（標準条件のみ）"
REPORT = f"""# 報酬重みの実験記録

仮説（実験前の計画）：{MY_HYPOTHESIS}
計画した変更：goal_progress、20 → {PROGRESS_WEIGHT}
実施状況：{comparison_status}
実行した条件：{my_results["condition"].unique().tolist()}
設定：{PROFILE}（demoはAPIの動作確認）
学習seed：{my_results["seed"].unique().tolist()}、最終評価seed：940000
PPOの初期化：ランダム。固定歩行モデルを使用。
各学習の総サンプル数：{options["steps"]}

## 結果
各学習seedの成功率・終了時距離：[記入]
複数学習の平均・標準偏差：[記入。学習1回の場合は未計算]
条件間の差：[記入。比較していない場合は未実施]
参照ファイル：experiment_results.csv、experiment_summary.csv、experiment_learning_curves.png、experiment_paired_results.png

## 仮説と照らし合わせた解釈
仮説を支持するか：[記入。比較していない場合は未判定]
判断の根拠となる数値・曲線・動画：[記入]

## この実験の限界
学習量・学習回数・初期配置の範囲による制約：[記入]

## 次に確かめる1項目
変更する値とその理由：[記入]
"""
(WORK_DIR / "report.md").write_text(REPORT)
print(REPORT)

### 実験結果を手元へ保存する

Colabの計算機内に置いたファイルは、ランタイムを破棄すると失われます。
次のセルは、`WORK_DIR`全体をZIPにまとめます。`DOWNLOAD_RESULTS=True`にして実行すると手元へダウンロードします。
ZIPには今回の実験記録が入ります。コードと実験データの取得はStep 0の`git clone`で行います。

In [ ]:
import shutil

archive = shutil.make_archive(str(WORK_DIR), "zip", root_dir=WORK_DIR.parent, base_dir=WORK_DIR.name)
print("実験結果ZIP:", archive)

DOWNLOAD_RESULTS = False  # 手元に保存するときは True
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(archive)

必要な観測・報酬・行動の項目は、次の付録で調べられます。
実験例の表・図・動画は、リポジトリの`results/`にあります。
[比較実験の詳細記録と学習済みモデル](https://github.com/hashimoto-robotics-lab/hexapod-transport-rl/tree/f85f3d754f18fe1efe57d724ffc23697e626c09f/results/reward_study)も参照できます。

## 付録 — 観測・報酬・行動のAPI一覧

必要な項目を探すための一覧です。最初に使う手順はStep 1・2・9・11・12で確認できます。
観測は`ObservationTerm(名前, scale=割る基準値)`で指定し、書いた順に連結します。
下の物理量は世界XY座標、距離はm、速度はm/s、角度はradです。

| 観測名 | 成分数 | 意味（scaleで割る前） |
|---|---:|---|
| `baseline` | 14 | Step 1.2の標準観測セット |
| `robot_to_cargo` | 2 | ロボットから円柱へのXYベクトル |
| `cargo_to_robot` | 2 | 円柱からロボットへのXYベクトル |
| `cargo_to_goal` | 2 | 円柱から目標へのXYベクトル |
| `robot_heading` | 2 | 機体の水平な向きの`[sin(yaw), cos(yaw)]` |
| `robot_velocity` | 2 | ロボットのXY速度 |
| `cargo_velocity` | 2 | 円柱のXY速度 |
| `robot_yaw_rate` | 1 | ロボットの旋回速度 [rad/s] |
| `cargo_yaw_rate` | 1 | 円柱の回転速度 [rad/s] |
| `goal_distance` | 1 | 円柱中心から目標までの距離 |
| `surface_gap` | 1 | ロボット中心から円柱外周までの距離。最小0 |
| `command_velocity` | 2 | 直前に要求した世界XY速度 |

`scale`の既定値は1です。例えば`ObservationTerm("cargo_velocity", scale=0.2)`は速度を0.2で割ります。
`enabled=False`なら同じ成分数のゼロを渡します。項を削除すると、ネットワークの入力層の大きさも変わります。
観測や行動の実験では、比較する両条件を`rotation="none"`にします。

標準の報酬7項はStep 2の表のとおりです。追加で次の組込項も使えます。

| 報酬名 | 重みを掛ける前の計算 | 時間の扱い |
|---|---|---|
| `cargo_speed` | 円柱のXY速度の大きさ | `dt`を掛ける |
| `command_change` | 前後の要求XY速度の差の二乗和 | `dt`を掛けない |

`rewards`に辞書を渡すと、辞書に含めた項だけを使います。
標準の他の項を残すには、Step 2のように`{**base.rewards, "項名": 重み}`でコピーします。
自作報酬は`RewardTerm(関数, weight=重み, per_second=時間を掛けるか)`で追加します。
関数は`(before, after)`を受け取り、世界ごとに1値、形`(B,)`を返します。実行例はStep 11です。

行動は次の設定から選べます。

| 設定 | 選べる操作 |
|---|---|
| `DiscreteDirections(count=4, include_stop=True)` | 停止と4方向の5択 |
| `DiscreteDirections(count=8, include_stop=True)` | 停止と8方向の9択 |
| `DiscreteDirections(count=16, include_stop=True)` | 停止と16方向の17択 |
| `ContinuousVelocity(frame="world", speed=0.15)` | 世界XYの連続した2出力 |
| `ContinuousVelocity(frame="body", speed=0.15)` | 機体の前方を+X、左側を+Yとする連続した2出力 |

離散行動の番号は+Xから反時計回りで、停止ありなら0が停止です。
`speed`は要求速度の上限、`frame`は行動を解釈する座標系です。観測の座標系は自動では変わりません。
連続行動は各軸[-1,1]の出力を、全体の大きさが1以下になるよう調整してから`speed`を掛けます。

自作の観測も、報酬と同じようにPythonファイルの直下へ関数を書けます。
`関数(state)`が返す形は`(B,)`または`(B,D)`で、`B`は並列世界数、`D`はその項の成分数です。
`ObservationTerm(関数, name="記録する名前", scale=割る基準値)`で指定します。
`PushState`の位置・速度などはStep 1とStep 11で使った`env.state`と同じ名前で読めます。
設定ファイルには`TASK = CylinderPushTask(...)`を定義し、Step 11の`load_task_file()`で読み込みます。